In [1]:
# Install necessary libraries
%pip install --quiet --upgrade \
    langchain-google-genai \
    langchain \
    langchain-community \
    langchain-text-splitters \
    chromadb \
    pypdf \
    pymupdf \
    unstructured \
    python-dotenv

print("All necessary libraries installed.")

Note: you may need to restart the kernel to use updated packages.
All necessary libraries installed.


In [2]:
# ========= 1. Paths & Imports =========
import os
import sys
from pathlib import Path
from typing import List

from dotenv import load_dotenv, find_dotenv
from pypdf import PdfReader   # If you no longer need to parse PDFs, you can delete it.

# --- Automatically infer the project root directory (avoid hardcoding absolute paths). ---
CWD = Path.cwd()

# Scenario 1: Open the notebook in the backend/prompts directory (recommended)
if CWD.name == "prompts" and CWD.parent.name == "backend":
    PROJECT_ROOT = CWD.parents[1]       # <project_root>/
# Scenario 2: Run directly in the project root directory (e.g., a Python script).
elif (CWD / "backend").exists():
    PROJECT_ROOT = CWD                  # <project_root>/
# Other situations: Go up another level to provide a safety net.
else:
    PROJECT_ROOT = CWD.parent

print("PROJECT_ROOT =", PROJECT_ROOT)

# Add the project root directory to sys.path for easier importing of backend.*
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Some commonly used subdirectories (which will be used later in auto_fill / load_docs)
BACKEND_DIR = PROJECT_ROOT / "backend"
DB_DIR      = BACKEND_DIR / "db"
PDF_DIR     = DB_DIR / "pdfs"
CHROMA_DIR  = PROJECT_ROOT / "chroma_db"

# Read the GOOGLE_API_KEY / DB configuration from the .env file.
load_dotenv(find_dotenv())

from backend.db.connection import get_db_connection

# LangChain Document
try:
    from langchain.schema import Document
except ModuleNotFoundError:
    from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma


PROJECT_ROOT = /Users/qizeng/Desktop/Thalia-AI


In [3]:
# ========= 2. Env & Config =========

load_dotenv(find_dotenv())

assert os.getenv("GOOGLE_API_KEY"), "GOOGLE_API_KEY 没读取到，请检查 .env"

DATA_DIR   = Path("./data")         # 如果未用到，可以之后删掉
CHROMA_DIR = Path("./chroma_db")

TOP_K        = 3
FETCH_K      = 60
MMR_LAMBDA   = 0.5
DOMAIN_TOPICS = [
    "menopause", "perimenopause", "postmenopause", "vasomotor", "hot flash", "night sweat",
    "hormone therapy", "HRT", "estradiol", "progesterone", "vaginal", "urogenital",
    "MRS", "menopause rating scale", "FSH", "AMH", "endometrium", "bone density",
    "Kupperman", "menopausal transition", "climacteric",
]
MIN_SIM_OUT_DOMAIN = 0.55
MIN_SIM_IN_DOMAIN  = 0.25


In [4]:
# ========= PDF helpers: metadata / authors / DOI =========
import re
from pathlib import Path
from typing import Optional
from pypdf import PdfReader

# --- Words used to identify "organization name" to avoid mistaking the organization for the author. ---
ORG_WORDS = re.compile(
    r"(?i)\b(University|Department|Center|Centre|Institute|Hospital|School|Faculty|"
    r"College|Laboratory|Society|Group|Clinic|Division)\b"
)

NAME_INIT_SURNAME = re.compile(r"(?:[A-Z]\.\s*){1,3}[A-Z][A-Za-z'\-]+")  # J. M. Smith
NAME_FULL = re.compile(
    r"\b[A-Z][a-z]{2,}(?:\s+[A-Z][a-z]{2,}){0,2}\s+[A-Z][a-z]{2,}\b"
)  # Jane M Smith

def _clean(s: Optional[str]) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

def _first_pages_text(pdf_path: Path, pages: int = 3, max_chars: int = 6000) -> str:
    """Use pypdf to extract the text from the first few pages."""
    text = []
    try:
        r = PdfReader(str(pdf_path))
        for i in range(min(pages, len(r.pages))):
            text.append(r.pages[i].extract_text() or "")
    except Exception:
        pass
    return _clean("\n".join(text))[:max_chars]

def _guess_year(*cands) -> str:
    """Find a year from a list of candidate strings, specifically 19xx or 20xx."""
    for x in cands:
        if not x:
            continue
        m = re.search(r"(19|20)\d{2}", x)
        if m:
            return m.group(0)
    return "n.d."

def _pdf_metadata(pdf_path: Path):
    """Retrieve the title and author from the PDF metadata."""
    try:
        info = PdfReader(str(pdf_path)).metadata
        title = _clean(getattr(info, "title", "") or "")
        author = _clean(getattr(info, "author", "") or "")
        return title, author
    except Exception:
        return "", ""

def _find_doi(page_text: str) -> Optional[str]:
    """Find DOI using simple regular expressions in the text."""
    m = re.search(r"(10\.\d{4,9}/[-._;()/:A-Za-z0-9]+)", page_text)
    return m.group(1) if m else None

def _extract_authors_from_text(page_text: str) -> str:
    """A simplified author extraction method without using fitz: Look for Initial + Surname near the title."""
    # 
    head = re.split(
        r"(?i)To cite this article|Abstract|Keywords|Introduction",
        page_text,
        maxsplit=1,
    )[0]
    lines = [l.strip() for l in head.split("\n") if l.strip()]

    # find title line
    title_idx = 0
    for i, l in enumerate(lines):
        if 10 <= len(l) <= 150 and not re.search(
            r"(?i)abstract|keywords|doi|reference", l
        ):
            title_idx = i
            break

    window = "\n".join(lines[title_idx + 1 : title_idx + 9])
    window = re.sub(r"[\d\*\u00B9\u00B2\u00B3]", "", window)
    window = (
        window.replace("’", "'")
        .replace("‐", "-")
        .replace("–", "-")
    )

    cand = NAME_INIT_SURNAME.findall(window)
    if 1 <= len(cand) <= 20:
        return ", ".join(dict.fromkeys([c.strip(" ,;&") for c in cand]))
    return ""

def _extract_authors_with_fitz(pdf_path: Path, max_lines_after_title: int = 10) -> str:
    """
    more detailed author retreive
    """
    try:
        import fitz  # PyMuPDF
    except Exception:
        return ""

    try:
        doc = fitz.open(str(pdf_path))
        if len(doc) == 0:
            doc.close()
            return ""
        page = doc[0]

        js = page.get_text("dict")  # blocks -> lines -> spans
        spans = []
        for b in js.get("blocks", []):
            for l in b.get("lines", []):
                for s in l.get("spans", []):
                    text = (s.get("text") or "").strip()
                    if not text:
                        continue
                    spans.append(
                        {
                            "text": text,
                            "size": float(s.get("size", 0)),
                            "y": float(s.get("bbox", [0, 0, 0, 0])[1]),
                            "line": l,
                        }
                    )

        if not spans:
            doc.close()
            return ""

        # 1) The largest font size, and unlike metadata, a span is approximately a title.
        candid = [
            sp
            for sp in spans
            if not re.search(r"(?i)abstract|keywords|doi|journal", sp["text"])
        ]
        if not candid:
            candid = spans[:]
        title_span = max(candid, key=lambda x: x["size"])
        title_y = title_span["y"]

        # 2) Select several rows below the title to serve as the author candidate window.
        below_lines = []
        seen_ids = set()
        for sp in sorted(spans, key=lambda x: (x["y"], -x["size"])):
            if sp["y"] <= title_y:
                continue
            lid = id(sp["line"])
            if lid not in seen_ids:
                seen_ids.add(lid)
                below_lines.append(sp["line"])
        window = "\n".join(
            " ".join(s.get("text", "") for s in ln.get("spans", [])).strip()
            for ln in below_lines[:max_lines_after_title]
        )

        window = re.sub(r"[\d\*\u00B9\u00B2\u00B3]", "", window)
        window = (
            window.replace("’", "'")
            .replace("‐", "-")
            .replace("–", "-")
        )
        window = re.sub(r"\s+", " ", window).strip()

        # 3) First, match the initials + last name; if that's not enough, then add the full name and filter out organizational terms.
        authors = []
        cand1 = NAME_INIT_SURNAME.findall(window)
        if len(cand1) >= 2:
            authors = cand1
        else:
            cand2 = [
                m
                for m in NAME_FULL.findall(window)
                if not ORG_WORDS.search(m)
            ]
            cand2 = [m for m in cand2 if 2 <= len(m.split()) <= 4]
            tmp = cand1 + cand2
            authors = list(
                dict.fromkeys([a.strip(" ,;&") for a in tmp if a.strip()])
            )

        doc.close()

        if 1 <= len(authors) <= 30:
            return ", ".join(authors)
        return ""
    except Exception:
        return ""


In [5]:
# ====== (This can be done manually when there are new PDFs and you want to update the database.) ======

PROJECT_ROOT = Path("/Users/qizeng/Desktop/Thalia-AI")
BACKEND_DIR  = PROJECT_ROOT / "backend"
DB_DIR       = BACKEND_DIR / "db"
PDF_DIR      = DB_DIR / "pdfs"


def auto_fill_db_with_fitz():
    """
    用 fitz + 文本自动补全 pdf_documents 里缺失的
    authors / doi / year，然后 UPDATE 回数据库。
    只需要偶尔手动跑一次这个函数。
    """
    conn = get_db_connection()
    cur  = conn.cursor(dictionary=True)

    # Only pick incomplete metadata
    cur.execute("""
        SELECT id, file_name, file_path, authors, doi, year, title
        FROM pdf_documents
        WHERE (authors IS NULL OR authors = '')
           OR (doi IS NULL OR doi = '')
           OR (year IS NULL OR year = 0)
    """)
    rows = cur.fetchall()

    print(f"Found {len(rows)} rows needing metadata fill")

    updates = []
    for row in rows:
        raw_path = row.get("file_path") or row["file_name"]
        p = Path(raw_path)

        pdf_path = None
        for cand in [
            p,
            BACKEND_DIR / raw_path,
            DB_DIR / raw_path,
            PDF_DIR / raw_path,
            PDF_DIR / row["file_name"],
        ]:
            if cand.exists():
                pdf_path = cand
                break

        if not pdf_path:
            print(f"[WARN] id={row['id']} pdf not found, skip")
            continue

        # 
        meta_title, meta_author = _pdf_metadata(pdf_path)
        page_text = _first_pages_text(pdf_path, pages=3)

        authors = (row["authors"] or "").strip()
        doi     = (row["doi"] or "").strip()
        year    = str(row["year"]) if row["year"] not in (None, "", 0) else ""

        if not authors:
            authors = (
                _extract_authors_with_fitz(pdf_path)
                or meta_author
                or _extract_authors_from_text(page_text)
                or ""
            )

        if not doi:
            doi = _find_doi(page_text) or ""

        if not year:
            year = _guess_year(meta_title, page_text, row.get("file_name"))

        # 
        if not authors and not doi and not year:
            continue

        updates.append((
            authors or None,
            doi or None,
            int(year) if (year and year.isdigit()) else None,
            row["id"],
        ))

        print(f"[UPDATE] id={row['id']} authors={authors!r} doi={doi!r} year={year!r}")

    # UPDATE to DB
    if updates:
        cur2 = conn.cursor()
        cur2.executemany(
            "UPDATE pdf_documents SET authors=%s, doi=%s, year=%s WHERE id=%s",
            updates
        )
        conn.commit()
        cur2.close()
        print(f"✅ updated {len(updates)} rows")
    else:
        print("No rows to update")

    cur.close()
    conn.close()


In [6]:
# ========= 3. Env & Config =========
from typing import List

def load_docs_from_db() -> List[Document]:
    conn = get_db_connection()
    cursor = conn.cursor(dictionary=True)

    cursor.execute("""
        SELECT
            id,
            file_name,
            title,
            authors,
            journal,
            year,
            doi,
            content,
            file_path
        FROM pdf_documents
        WHERE content IS NOT NULL AND content <> ''
    """)
    rows = cursor.fetchall()
    cursor.close()
    conn.close()

    docs: List[Document] = []

    for i, row in enumerate(rows):
        # ---------- 1. Find the actual PDF path----------
        raw_path = row.get("file_path") or row["file_name"]
        p = Path(raw_path)

        pdf_path: Path | None = None

        # Scenario 1: The absolute path is already stored in the database.
        if p.is_absolute() and p.exists():
            pdf_path = p
        else:
            # Scenario 2: Several possible relative paths (try them one by one)
            candidates = [
                BACKEND_DIR / raw_path,
                DB_DIR / raw_path,
                PDF_DIR / raw_path,
                PDF_DIR / row["file_name"],  
            ]
            for c in candidates:
                if c.exists():
                    pdf_path = c
                    break

        # Print out the first few items to help  confirm which PDF found.
        if i < 3:
            print(f"[DEBUG] id={row['id']}, file_name={row['file_name']}")
            print(f"        raw_path = {raw_path}")
            print(f"        resolved pdf_path = {pdf_path}")
            print("")

        # ---------- 2. First use the metadata in the DB ----------
        authors = row["authors"] or ""
        year    = str(row["year"]) if row["year"] not in (None, "", 0) else ""
        title   = row["title"] or row["file_name"]
        journal = row["journal"] or ""
        doi     = row["doi"] or ""

        # ---------- 3.If anything is missing, add it from the PDF.  ----------
        page_text = ""
        if pdf_path is not None and pdf_path.exists():
            meta_title, meta_author = _pdf_metadata(pdf_path)
            page_text = _first_pages_text(pdf_path, pages=3)

            if not title and meta_title:
                title = meta_title

            if not authors:
                authors = (
                    _extract_authors_with_fitz(pdf_path)
                    or meta_author
                    or _extract_authors_from_text(page_text)
                    or "Unknown"
                )

            if not doi:
                doi = _find_doi(page_text) or ""

            if not year:
                year = _guess_year(meta_title, page_text, row.get("file_name"))

        # ---------- 4.  ----------
        if not authors:
            authors = "Unknown"
        if not year:
            year = "n.d."
        if not title:
            title = row["file_name"]

        # ---------- 5. citation ----------
        if doi:
            cit = f"{authors}. ({year}). {title}. {journal}. https://doi.org/{doi}"
        else:
            cit = f"{authors}. ({year}). {title}. {journal}."

        metadata = {
            "source":    f"db:pdf_documents:{row['id']}",
            "file_name": row["file_name"],
            "file_path": str(pdf_path) if pdf_path else row.get("file_path"),
            "title":     title,
            "authors":   authors,
            "journal":   journal,
            "year":      year,
            "doi":       doi,
            "citation":  cit,
        }

        docs.append(
            Document(
                page_content=row["content"],
                metadata=metadata,
            )
        )

    print(f"Loaded {len(docs)} docs from DB")
    return docs


In [7]:
#Test for the presence of DOI (can be deleted).
docs = load_docs_from_db()
print(docs[0].metadata)   


[DEBUG] id=1, file_name=2023-practitioner-toolkit-for-managing-menopause-2023-islam.pdf
        raw_path = /Users/qizeng/Desktop/Thalia-AI/backend/db/pdfs/2023-practitioner-toolkit-for-managing-menopause-2023-islam.pdf
        resolved pdf_path = /Users/qizeng/Desktop/Thalia-AI/backend/db/pdfs/2023-practitioner-toolkit-for-managing-menopause-2023-islam.pdf

[DEBUG] id=2, file_name=BMC-Women-knowledge-attitudes-to-menopause-harper.pdf
        raw_path = /Users/qizeng/Desktop/Thalia-AI/backend/db/pdfs/BMC-Women-knowledge-attitudes-to-menopause-harper.pdf
        resolved pdf_path = /Users/qizeng/Desktop/Thalia-AI/backend/db/pdfs/BMC-Women-knowledge-attitudes-to-menopause-harper.pdf

[DEBUG] id=3, file_name=a-review-of-menopause-nomenclature-2022.pdf
        raw_path = /Users/qizeng/Desktop/Thalia-AI/backend/db/pdfs/a-review-of-menopause-nomenclature-2022.pdf
        resolved pdf_path = /Users/qizeng/Desktop/Thalia-AI/backend/db/pdfs/a-review-of-menopause-nomenclature-2022.pdf

Loaded 34 

In [8]:
# ========= 4. Split into chunks & build Chroma =========
#
#print(docs[0].metadata)

# 4.1 Load documents from the database
docs = load_docs_from_db()

# 4.2 Cut into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
texts = text_splitter.split_documents(docs)
print("Number of chunks:", len(texts))

# 4.3 （Gemini Embeddings + Chroma）
emb = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

CHROMA_DIR.mkdir(exist_ok=True)
vectordb = Chroma.from_documents(
    documents=texts,
    embedding=emb,
    persist_directory=str(CHROMA_DIR),
)

print("Vector store saved to:", CHROMA_DIR.resolve())


[DEBUG] id=1, file_name=2023-practitioner-toolkit-for-managing-menopause-2023-islam.pdf
        raw_path = /Users/qizeng/Desktop/Thalia-AI/backend/db/pdfs/2023-practitioner-toolkit-for-managing-menopause-2023-islam.pdf
        resolved pdf_path = /Users/qizeng/Desktop/Thalia-AI/backend/db/pdfs/2023-practitioner-toolkit-for-managing-menopause-2023-islam.pdf

[DEBUG] id=2, file_name=BMC-Women-knowledge-attitudes-to-menopause-harper.pdf
        raw_path = /Users/qizeng/Desktop/Thalia-AI/backend/db/pdfs/BMC-Women-knowledge-attitudes-to-menopause-harper.pdf
        resolved pdf_path = /Users/qizeng/Desktop/Thalia-AI/backend/db/pdfs/BMC-Women-knowledge-attitudes-to-menopause-harper.pdf

[DEBUG] id=3, file_name=a-review-of-menopause-nomenclature-2022.pdf
        raw_path = /Users/qizeng/Desktop/Thalia-AI/backend/db/pdfs/a-review-of-menopause-nomenclature-2022.pdf
        resolved pdf_path = /Users/qizeng/Desktop/Thalia-AI/backend/db/pdfs/a-review-of-menopause-nomenclature-2022.pdf

Loaded 34 

In [9]:
# ========= 5. Domain & score utils =========
import re

DOMAIN_RE = re.compile("|".join(re.escape(k) for k in DOMAIN_TOPICS), re.I)

def _looks_in_domain(q: str) -> bool:
    """A rough assessment is needed to determine if the problem is related to menopause."""
    return bool(DOMAIN_RE.search(q))

def _scores_from_results(results):
    """
    results: List[(Document, distance)]  ->  List[(Document, similarity)]
    Convert the "distance" returned by Chroma into a "similarity" between 0 and 1.
    """
    sims = []
    for d, dist in results:
        try:
            sim = 1.0 - float(dist)      # The smaller the distance, the higher the similarity.
        except Exception:
            sim = 0.0
        sims.append((d, max(0.0, min(1.0, sim))))
    return sims


In [10]:
# ========= 6. Citation helpers =========
def _apa_like_from_metadata(meta: dict) -> str:
    author  = meta.get("authors") or "Unknown"
    year    = str(meta.get("year") or "n.d.")
    title   = meta.get("title") or meta.get("file_name") or "Untitled"
    journal = meta.get("journal") or ""
    doi     = meta.get("doi") or ""

    parts = [f"{author}. ({year}). {title}."]
    if journal:
        parts.append(journal + ".")
    if doi:
        if doi.startswith("10."):
            parts.append(f"https://doi.org/{doi}")
        else:
            parts.append(doi)
    return " ".join(parts)

def numbered_citation(doc, idx: int) -> str:
    meta = doc.metadata or {}
    if meta.get("citation"):
        return f"[{idx}] {meta['citation']}"
    return f"[{idx}] {_apa_like_from_metadata(meta)}"


In [11]:
MENOPAUSE_RAG_TEMPLATE = """You are a compassionate and knowledgeable menopause support assistant.

Before answering, please:
1. Analyze the user's query to understand their core intent
2. If the query is vague or uses colloquial terms, mentally rephrase it using proper medical terminology
3. Consider related menopause symptoms or concerns that might be relevant
4. If the query is incomplete, use your menopause knowledge to provide comprehensive context

You provide responses in different helpful tones depending on the situation.

---

[Expert Style Example]

Context:
Hot flashes are sudden sensations of intense heat due to changes in thermoregulation linked to declining estrogen.

Question:
Why do I keep getting hot flashes?

Answer:
Hot flashes are a common vasomotor symptom of menopause caused by disruptions in the hypothalamus, which regulates body temperature. These changes are primarily driven by declining estrogen levels. The body responds by dilating blood vessels and triggering sweating as a cooling mechanism. If they interfere with your quality of life, consult a healthcare provider to explore treatment options, including hormone therapy or non-hormonal alternatives.

---

[Warm Empathetic Style Example]

Context:
Sleep problems and night sweats are discussed in the context of menopause transition.

Question:
I can't sleep and feel exhausted all the time. Is this normal?

Answer:
I'm so sorry you're feeling this way — it's completely understandable. Many women experience sleep issues during menopause, especially due to night sweats or hormonal changes. You're not alone. One step that might help is keeping your bedroom cool, using breathable sheets, and tracking your symptoms. Please know that support is available, and you're doing your best in a challenging time. You're not alone.

---

[Blended Style Example]

Context:
User describes feeling emotionally overwhelmed, confused, and unsure how to cope with menopausal symptoms.

Question:
I'm scared and confused about these symptoms. Why do I feel so lost during menopause?

Answer:
It's completely understandable to feel lost during menopause — your body is going through major changes, and it can affect your emotions, energy, and clarity. Many women feel overwhelmed at this stage. On a medical level, hormone shifts — especially the drop in estrogen — can impact mood regulation, sleep, and mental clarity. If you feel ready, journaling your symptoms and talking to a supportive doctor can really help. You're not alone — we can take it one step at a time.

---

[Expert Style Example #2]

Context:
Estrogen levels drop significantly during menopause.

Question:
What does low estrogen actually do to the body?

Answer:
Low estrogen during menopause can affect multiple systems — it may lead to symptoms like hot flashes, vaginal dryness, mood changes, and decreased bone density. Estrogen helps regulate temperature, maintains urogenital tissue health, and protects against osteoporosis. If symptoms are severe, options like hormone replacement therapy (HRT) may be discussed with your doctor.

---

[Warm Empathetic Style Example #2]

Context:
User shares that they feel alone and unsupported during menopause.

Question:
Why does no one talk about menopause? I feel like I'm going through this alone.

Answer:
You're absolutely not alone, even if it feels that way. So many women go through this transition silently because it's not talked about enough. But your experience is real and valid. It's okay to feel confused or isolated. There are supportive communities, resources, and health professionals who truly care — you deserve that support. Thank you for speaking up.

---

Now, based on the following context and user question, please respond in a helpful, appropriate tone (expert, warm, or blended):

Context:
{context}

Question:
{question}

Answer:"""


In [12]:
# ==== 6. Short-answer generator (Gemini + menopause template) ====
import os
import google.generativeai as genai

def _gen_short_answer(context_snips, question, in_domain, top_sim, max_chars=800):
    """
    Based on the retrieved context snippets and the user's question, generate a short answer using Gemini. 
    Preferably use MENOPAUSE_RAG_TEMPLATE; if Gemini encounters an error, use a catch-all text.
    """
    ctx = "\n\n".join(context_snips)

    try:
        key = os.getenv("GOOGLE_API_KEY")
        if not key:
            raise RuntimeError("GOOGLE_API_KEY not set")

        genai.configure(api_key=key)
        model = genai.GenerativeModel("gemini-2.0-flash")

        # 用你定义好的 template
        prompt = MENOPAUSE_RAG_TEMPLATE.format(
            context=ctx,
            question=question,
        )

        resp = model.generate_content(prompt)
        text = (resp.text or "").strip()
        if not text:
            raise RuntimeError("Empty response from Gemini")

        # Normal situation: Return the real answer
        return text[:max_chars] if max_chars else text

    except Exception as e:
        # Print out the actual error so you can troubleshoot it.
        print("⚠️ Gemini call failed. Here's a fallback answer: ", repr(e))

        # The decision to reject or weakly answer is based on in_domain + top_sim.
        if (not in_domain) and (top_sim < MIN_SIM_OUT_DOMAIN):
            return (
                "This question appears outside the menopause domain of your local library. "
                "Please ask a menopause-related question."
            )

        if in_domain and (top_sim < MIN_SIM_IN_DOMAIN):
            return (
                "I found some potentially related sources in your menopause library. "
                "If it's not specific enough, try a narrower query."
            )

        # 
        return "I found relevant professional sources for your question and listed them below."


/Users/qizeng/anaconda3/envs/lcenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
# ========= 7. Chat (Answer + Citations) =========
def chat(question: str, top_k: int | None = None, use_mmr: bool = True) -> str:
    q = question.strip()
    assert "vectordb" in globals(), "vectordb is not defined. Please build the vector library first."
    if top_k is None:
        top_k = TOP_K

    in_domain = _looks_in_domain(q)

    # 1) Search: MMR + similarity_search (fallback)
    CAND_MULT   = 3
    MMR_FETCH_K = max(FETCH_K, top_k * 8)

    try:
        mmr_hits = []
        if use_mmr:
            retriever = vectordb.as_retriever(
                search_type="mmr",
                search_kwargs={
                    "k": top_k * CAND_MULT,
                    "fetch_k": MMR_FETCH_K,
                    "lambda_mult": MMR_LAMBDA,
                },
            )
            mmr_hits = retriever.invoke(q)

        try:
            scored = vectordb.similarity_search_with_score(q, k=top_k * CAND_MULT)
            if not scored and mmr_hits:
                scored = [(d, 1.0) for d in mmr_hits]
        except Exception:
            base = mmr_hits or vectordb.similarity_search(q, k=top_k * CAND_MULT)
            scored = [(d, 1.0) for d in base]
    except Exception:
        base = vectordb.similarity_search(q, k=top_k * CAND_MULT)
        scored = [(d, 1.0) for d in base]

    sims = _scores_from_results(scored)
    if not sims:
        msg = "🌸 Dear:\n\nNo documents found in your local menopause library."
        print(msg)
        return msg

    sims.sort(key=lambda x: x[1], reverse=True)
    top_sim = sims[0][1]

    # 2) Only those with "strong out-of-domain + low similarity" will be directly rejected.
    if (not in_domain) and (top_sim < MIN_SIM_OUT_DOMAIN):
        msg = (
            "🌸 Dear:\n\n"
            "This question appears outside the menopause domain of your local library. "
            "Please ask a menopause-related question."
        )
        print(msg)
        return msg

    # 3) Choose the documents to use for the final answer (prioritizing different sources).
    docs, seen_src = [], set()
    for d, _ in sims:
        src = d.metadata.get("source", "")
        if src in seen_src:
            continue
        seen_src.add(src)
        docs.append(d)
        if len(docs) >= top_k:
            break
    if len(docs) < top_k:
        for d, _ in sims:
            if d in docs:
                continue
            docs.append(d)
            if len(docs) >= top_k:
                break

    # 4) Use the  _gen_short_answer above + MENOPAUSE_RAG_TEMPLATE
    ctx_snips = [d.page_content[:700] for d in docs[:min(4, len(docs))]]
    answer = _gen_short_answer(ctx_snips, q, in_domain, top_sim)

    # 5) Citations: Deduplicated by doi/title/source, then numbered.
    unique_docs = []
    seen_keys = set()
    for d in docs:
        md_meta = d.metadata or {}
        key = (
            md_meta.get("doi")
            or md_meta.get("title")
            or md_meta.get("source")
            or md_meta.get("file_path")
        )
        if not key:
            key = d.page_content[:100]
        if key in seen_keys:
            continue
        seen_keys.add(key)
        unique_docs.append(d)

    refs = [numbered_citation(d, i + 1) for i, d in enumerate(unique_docs)]

    md = (
        "🌸 Dear:\n\n"
        f"{answer}\n\n"
        "References:\n\n" + "\n".join(f"— {r}" for r in refs)
    )
    print(md)
    #return md


In [14]:
#Call to update the database (optional)
auto_fill_db_with_fitz()

Found 8 rows needing metadata fill
[UPDATE] id=4 authors='Martha Hickey, Jennifer Doust, Muthusamy Sivakami, Deborah Garlick, Hunter Menopause, Published Online March' doi='' year='2024'
[UPDATE] id=14 authors='Menopause Sultanate, Oman Ministry, Health Directorate General, Community Health' doi='' year='2010'
[UPDATE] id=15 authors='' doi='10.4103/jmh.JMH_96_18' year='2018'
[UPDATE] id=17 authors='Promoting Lifestyle Education Intervention, Promoting Behaviors, Health Status, Postmenopausal Women, Experimental Study, Sri Lanka, Gayani Alwis, Janaka Lenora, Sarath Lekamwasam Nirmala Rathnayake, Allied Health Sciences' doi='' year='2019'
[UPDATE] id=25 authors='Postmenopausal Women Karishma Sharin, Shanmuganath Elayaperumal, Dhivya Bharathi Annadurai Research, Sri Balaji Vidyapeeth, Research Scholar, Corresponding Author' doi='' year='2025'
[UPDATE] id=26 authors='Artificial Intelligence Introduction Menopause, Atul Munshi' doi='' year='2024'
[UPDATE] id=30 authors='' doi='' year='n.d.'

In [15]:
# Q1
chat("What is menopause?")

🌸 Dear:

Menopause is a significant biological event in a woman's life, defined as the permanent end of menstruation, typically occurring in the late 40s or early 50s. This transition is marked by the ovaries ceasing to release eggs and a substantial decline in the production of estrogen and progesterone. These hormonal shifts can lead to various physical, emotional, and cognitive changes. Recent research continues to improve our understanding of these changes, paving the way for more effective management strategies.

References:

— [1] Unknown. (2024). the-impact-of-menopause-on-women-health-a-review-of-recent-research-2024. .
— [2] S. Bhuvaneswari, M. Sc, S.Bhuvaneswari, M. Sc. (2024). the-impact-of-menopause-on-women-health-a-review-of-recent-research-2024. . https://doi.org/10.47997/sdes-ijir/5.5.2024:844-847


In [16]:
#Q2
chat("What is the MRS?")

🌸 Dear:

The Menopause Rating Scale (MRS) is a widely used tool to assess the severity and frequency of menopause symptoms. Developed by Heinmann et al. in 2000, it consists of 11 items, and is used in clinical and research settings to evaluate the impact of menopause on women's health.

References:

— [1] Unknown. (2016). the-effectiveness-of-lifestyle-educational-program-in-health-promoting-in-health-promoting-behaviors-menopausal-symptoms-45-60-year-old-women-Iran-2016. .
— [2] Promoting Behaviors, Menopausal Symptoms, Old Women, Iran Mahin Nazari, Samaneh Farmani, Mohammad Hosein Kaveh, Health Education, Health Promotion, Medical Sciences, Iran Correspondence, Mahin Nazari, Online Published. (2016). the-effectiveness-of-lifestyle-educational-program-in-health-promoting-in-health-promoting-behaviors-menopausal-symptoms-45-60-year-old-women-Iran-2016. . https://doi.org/10.5539/gjhs.v8n10p34


In [17]:
chat("how to learn calculus?")

🌸 Dear:

This question appears outside the menopause domain of your local library. Please ask a menopause-related question.


'🌸 Dear:\n\nThis question appears outside the menopause domain of your local library. Please ask a menopause-related question.'

In [18]:
chat("what is the difference between menopause and premenopause?")

🌸 Dear:

Okay, I understand. The user is asking about the distinction between menopause and perimenopause. Based on the provided text and my knowledge, I'll provide a blended style answer defining both terms and explaining their differences, touching on age-related criteria and common experiences.

Here's my response:

Okay, let's clarify the difference between perimenopause and menopause. It's a common point of confusion!

**Perimenopause** is the transitional phase leading up to menopause. Think of it as the "menopause journey." During this time, your ovaries gradually produce less estrogen, but it's not a consistent decline. Estrogen levels can fluctuate significantly, which is why you might experience irregular periods and other symptoms like hot flashes, sleep disturbances, and mood changes. A

References:

— [1] Ananthan Ambikairajah, Erin Walsh, Nicolas Cherbuin Abstract Menopause, Aging Workshop. (2022). a-review-of-menopause-nomenclature-2022. . https://doi.org/10.1186/s12978-

In [19]:
chat("What’s the difference between menopause and perimenopause?")

🌸 Dear:

Okay, I understand. The user wants to know the difference between menopause and perimenopause, and the context provides definitions for both. I'll aim for a blended style, explaining the difference clearly while also acknowledging that this can be a confusing time.

Here's my response:

"It's a great question, and it's common to be a little confused about the difference between menopause and perimenopause. Think of it this way: menopause is really just one specific day – it's defined as the day 12 months after your last period. Perimenopause, on the other hand, is the *years leading up to* that final period.

During perimenopause, which typically starts in your 40s and can last anywhere from 1 to 10 years, your periods become irregular, and you might start experiencing symptoms like hot fl

References:

— [1] Unknown. (2025). women-experiences-expectations-during-menopause-transition-systematic-qualitative-narrative-review-wood-2025.
— [2] L. Thomas, Katrina Wood, Hannah Pitt,